# NB07_fPINN — Long-horizon accuracy, sum-of-exponentials competitor and fast L1 cost of the finite-memory realizations

**Purpose.** Second notebook of the finite-memory study. It corrects the window + tail code of the first NB06 run (E0) and extends NB06 in three directions (Q1–Q3):

| Item | Question | Experiment here |
|---|---|---|
| **E0** | Correction found while writing NB07: the `+ tail` (mean) correction of NB06 averaged the wrong samples. | Old vs corrected offline error on the NB06 references; every `+ tail` realization is rerun in the closed loop of E2. |
| **Q1** | Accuracy was only shown up to 60 s, but the controller must run indefinitely. Oustaloup cannot forget a displacement held below its band (`H(0) = ω_b^α`). | **E1**: offline (no networks) 60 min references with pauses at the ready pose, pauses at a displaced pick pose and work away from the initial pose; error of every realization against the L1 sum, minute by minute. **E1b**: analytic DC limit and step-error curve of Oustaloup, checked against the holds of E1. |
| **Q2** | Is a sum-of-exponentials (SOE) realization a better fixed-cost competitor? | **E2**: SOE realization of the L1 history (kernel `τ^{−α}/Γ(1−α)` replaced by `Σ w_i e^{−λ_i τ}` with relative error ≤ ε on `[T_C, T_hor]`, trapezoidal rule on `τ^{−α} = ∫ e^{−τ e^x + αx} dx / Γ(α)`, in the spirit of Jiang et al. 2017; local step exact) in E1 (offline, 60 min), in the closed loop of NB06 (oracle, fPINN and PINN, same scenarios, seeds and gain-kept metric) and in E3. |
| **Q3** | The cost argument used the naive O(n) L1 sum in Python, and the C kernels were not measured. | **E3a**: Python cost per update of the naive L1, an **exact fast L1** (block FFT: old history by one FFT per block of L steps, recent samples directly; mean and *worst-case* step), window + tail, Oustaloup and SOE, from 1 s to 1 h of operation. **E3b**: the same fixed-cost kernels and the naive L1 in plain C99, compiled and timed if a C compiler is found (gcc, clang or cc); otherwise the `.c` file is written and the step is skipped. |

The plant, controller, references, networks and the closed-loop code are those of NB06 (copied here, unchanged), so the closed-loop rows of NB07 for `L1 full` and `Oustaloup 7 [1e-2, 1e3]` must reproduce NB06 (checked if `NB06_fPINN_ClosedLoop.csv` is in the folder).

**One switch.** `QUICK = True` is a fast check (~1 min: 3 min references, α = 0.5, closed loop with the oracle and one network, short timing). `QUICK = False` runs **the whole experiment set of the article** with nothing else to edit.

**Expected runtime on the Ryzen 7 5850U: about 1 h 10 min (50 min to 1 h 40 min).** E0 ≈ 2 min; E1 + E1b ≈ 12 min (vectorized: 4 references × 1.8 million steps × 4 orders × 16 realizations; 6.5 min measured on the development machine, ×1.5); E2 closed loop ≈ 50 min (23 batches of 8 realizations and 19 PINN batches, × 3 scenarios; ≈ 33 min estimated on the development machine, ×1.5); E3a ≈ 1 min; E3b < 1 min if a compiler exists. The progress line of E2 prints the time left. E1 and E2 are appended to their CSVs as they go and skipped on a rerun, so an interruption loses at most one reference or one batch. RAM < 3 GB.

**Put in the same folder:** this notebook, the `NB02_fPINN_*_e150.npz` files of NB02 and the folder `NB04_fPINN_weights` of NB04 (as for NB06). Optional: `NB06_fPINN_ClosedLoop.csv` (consistency check).

**Outputs:** `NB07_fPINN_CorreccionCola.csv`, `NB07_fPINN_LargoPorMinuto.csv`, `NB07_fPINN_LargoResumen.csv`, `NB07_fPINN_CotaOustaloup.csv`, `NB07_fPINN_SOE.csv`, `NB07_fPINN_ClosedLoop.csv`, `NB07_fPINN_Summary.csv`, `NB07_fPINN_Timing.csv`, `NB07_fPINN_TimingC.csv` (only if a compiler was found), `fig_nb07_*.png/pdf`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/Anaconda: same fix as NB04/NB06
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[_v] = "1"              # one BLAS thread (as on a controller); must be set before NumPy loads: restart the kernel first
import time, shutil, subprocess, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve, lfilter
from scipy import stats
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------------------------
QUICK        = True                # True: fast check (~2 min). False: the whole experiment set of the article (see header).
B_FRAC       = 0.05                # [N m s^α] per joint (as NB02-NB06)
ALPHA_MAIN, MAIN_SEEDS = 0.5, tuple(range(10))
ALPHA_SWEEP, SWEEP_SEEDS = (0.3, 0.7, 0.9), (0, 1, 2)
SIZE, EPOCHS_TAG = 175, 150
SINGLE_SCALES = (1.6, 3.2)         # closed loop: one Adept cycle (4 s, 8 s), 0 kg, as NB06
CONT_SCALE, N_CYCLES = 1.6, 15     # closed loop: continuous operation 15 x 4 s = 60 s, as NB06
PAYLOAD      = 0.0
NB02_WEIGHTS = "."
NB04_WEIGHTS = "NB04_fPINN_weights"
NB06_CL_CSV  = "NB06_fPINN_ClosedLoop.csv"
FS = 2000; dt = 1.0/FS; T_C = 0.002; STEP_C = int(round(T_C/dt))
TAU_MAX = 5.0
LONG_MIN     = 60                  # E1: length of the long references [min]
ALPHAS_OFF   = (0.3, 0.5, 0.7, 0.9)
SOE_HOR      = 3600.0              # E2: SOE horizon T_hor [s] (relative kernel error <= eps on [T_C, T_hor])
SOE_EPS      = (1e-2, 1e-3, 1e-4)
TIMING_T     = (1, 10, 60, 600, 1200, 3600)   # E3: elapsed operation time [s]
BLOCK_L      = (256, 1024, 4096)   # E3: block lengths of the fast (block-FFT) L1
if QUICK:
    MAIN_SEEDS, ALPHA_SWEEP, SINGLE_SCALES, N_CYCLES = (0,), (), (1.6,), 3
    LONG_MIN, ALPHAS_OFF, TIMING_T, BLOCK_L = 3, (0.5,), (1, 10, 60), (1024,)
SUFFIX = "_QUICK" if QUICK else ""
OUT_CL = f"NB07_fPINN_ClosedLoop{SUFFIX}.csv"
fmt_eps = lambda e: f"{e:.0e}".replace("e-0", "e-")
fmt_hor = lambda T: f"{T/3600:g} h" if T >= 3600 else f"{T:g} s"

def realizations():
    """Closed loop (E2): L1 full (reference), Oustaloup 7 [1e-2, 1e3] (NB06 choice, consistency check), corrected windows + tail and SOE."""
    R = [("L1 full", "l1", {}), ("Oustaloup 7 [1e-2, 1e3]", "oust", dict(n=7, wb=1e-2, wh=1e3))]
    R += [(f"window {tw:g} s + tail", "win", dict(Tw=tw, tail="mean")) for tw in (0.1, 0.5, 2.0)]   # corrected code (E0)
    R += [(f"SOE {fmt_eps(e)} [{fmt_hor(SOE_HOR)}]", "soe", dict(eps=e, T=SOE_HOR)) for e in SOE_EPS]
    if QUICK: R = [r for r in R if r[0] in ("L1 full", "Oustaloup 7 [1e-2, 1e3]", "window 0.5 s + tail") or r[2].get("eps") == 1e-3]
    return R
REALIZ = realizations()
print("QUICK" if QUICK else "FULL", f"| {len(REALIZ)} realizations in closed loop: {[r[0] for r in REALIZ]}")

## Robot model and plant (NB03/NB04, written for a batch of robots)

Every function takes arrays with a leading batch axis `(M, 3)`; with `M = 1` the arithmetic is that of NB03. The plant's fractional damper uses the **full-memory L1 sum at 2 kHz** for every member; to keep 60 s of history affordable it is evaluated in blocks (old samples by FFT once per block, recent ones directly), which is the same sum up to rounding (checked below).

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)
def gamma_payload(m): return (M_BASE + float(m)) / M_BASE

def fk_b(q):
    """Forward kinematics of NB03 for q (M, 3) -> p (M, 3)."""
    rd = R_BASE - R_PLAT; C = np.zeros((len(q), 3, 3))
    for i in range(3):
        a = 2*i*np.pi/3; r = rd + L_A*np.cos(q[:, i])
        C[:, i, 0] = r*np.cos(a); C[:, i, 1] = r*np.sin(a); C[:, i, 2] = L_A*np.sin(q[:, i])
    d1 = 2*(C[:, 1] - C[:, 0]); d2 = 2*(C[:, 2] - C[:, 0]); A1, B1, C1 = d1.T; A2, B2, C2 = d2.T
    D1 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 1])); D2 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 2]))
    den = A2*B1 - A1*B2
    L1 = (A1*D2 - A2*D1)/den; L2 = (A1*C2 - A2*C1)/den; L3 = (B2*D1 - B1*D2)/den; L4 = (B2*C1 - B1*C2)/den
    u = 1 + L2**2 + L4**2; c0 = C[:, 0]
    v = 2*(L3*L4 + L1*L2 - c0[:, 0]*L4 - c0[:, 1]*L2 - c0[:, 2])
    w = c0[:, 0]**2 + c0[:, 1]**2 + c0[:, 2]**2 + L3**2 + L1**2 - 2*c0[:, 0]*L3 - 2*c0[:, 1]*L1 - L_B**2
    z = (-v + np.sqrt(v**2 - 4*u*w)) / (2*u)
    return np.column_stack([L3 + L4*z, L1 + L2*z, z])

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def regressor_b(q, a, p):
    """(M, 3, 14) regressor of NB01 for a batch."""
    x, y, z = p[:, 0], p[:, 1], p[:, 2]; s, c = np.sin(q), np.cos(q); Y = np.zeros((len(q), 3, 14))
    Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y

def tau_unmodeled_b(q, qd):
    u = UNMODELED
    return (u["b_v"]*qd + u["f_c"]*np.tanh(qd/u["eps"])
            + u["c_cor"]*(qd**2*np.cos(q) + qd[:, [1, 2, 0]]*qd[:, [2, 0, 1]]) + u["k_pos"]*np.sin(u["w_pos"]*q))

def plant_qdd_b(q, qd, tau, p, m, Dq):
    g = gamma_payload(m)
    return (tau - g*(regressor_b(q, np.zeros_like(q), p) @ THETA0) - tau_unmodeled_b(q, qd) - B_FRAC*Dq) / (g*THETA0[0])

def F_nominal_b(q, qd, qdd, m, p): return gamma_payload(m)*(regressor_b(q, qdd, p) @ THETA0)
def F_oracle_b(q, qd, qdd, m, p):  return F_nominal_b(q, qd, qdd, m, p) + tau_unmodeled_b(q, qd)

def l1_weights(alpha, n): k = np.arange(n, dtype=float); return (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
def caputo_l1(x, h, alpha):
    """Batch L1 Caputo derivative of x (N, c) from t = 0 (reference)."""
    N = len(x); d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    return fftconvolve(d, l1_weights(alpha, N)[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)

class PlantL1:
    """Full-memory L1 Caputo derivative of C channels sampled every h from t = 0, evaluated in blocks of L samples:
    samples older than the block by one FFT convolution per block, the rest by a direct sum. Same sum as NB03's FracHistory."""
    def __init__(self, alpha, h, n_max, C, L=1000):
        self.b = l1_weights(alpha, n_max + L + 2); self.c = h**(-alpha) / Gamma(2 - alpha); self.L = L
        self.br = self.b[:L][::-1].copy(); self.dq = np.zeros((n_max + 2, C)); self.n = 0; self.q_last = None; self.far = np.zeros((L, C))
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros_like(q)
        self.n += 1; n = self.n; self.dq[n] = q - self.q_last; self.q_last = q.copy()
        if (n - 1) % self.L == 0:                     # new block starting at n0 = n: contribution of dq_1 ... dq_{n0-1}
            self.n0 = n
            if n > 1: self.far = fftconvolve(self.dq[1:n], self.b[:n - 1 + self.L][:, None], axes=0)[n - 1:n - 1 + self.L]
        j = n - self.n0
        return self.c * (self.far[j] + self.br[self.L - j - 1:] @ self.dq[self.n0:n + 1])

## Realizations of the controller's memory term `D^α̂ q`, updated every `T_C` from the measured angles

All share `reset()` and `push(q) -> D^α̂ q(t_n)` (3-vector), with t = 0 at the start of the run, as in NB03/NB04. `MemWindow` is the corrected window + tail (see E0 below); `MemWindowNB06` is the code of the first NB06 run, kept only for E0.

In [ ]:
class MemNone:
    def __init__(self, alpha): pass
    def reset(self, n_max): pass
    def push(self, q): return np.zeros(3)

class MemL1:
    """Full memory, direct sum (what a controller would execute; = FracHistory of NB03, single-threaded)."""
    def __init__(self, alpha, h=T_C): self.alpha, self.h = alpha, h; self.c = h**(-alpha)/Gamma(2 - alpha)
    def reset(self, n_max):
        self.b = l1_weights(self.alpha, n_max + 1)[::-1].copy(); self.dq = np.zeros((n_max + 1, 3)); self.n = 0; self.q_last = None
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros(3)
        self.n += 1; self.dq[self.n] = q - self.q_last; self.q_last = q.copy()
        return self.c * np.einsum("i,ij->j", self.b[-self.n:], self.dq[1:self.n + 1])   # einsum: no BLAS call, so no thread overhead

class MemWindowNB06:
    """NB06 code, kept only to quantify its error (see the correction below). L1 sum over the last N_w = T_w/h increments (ring buffer), plus an O(1) correction of the discarded part
    (summation by parts, q̃ = q − q_0): tail='lead' adds b_Nw·q̃_{n−Nw}; tail='mean' also subtracts (b_Nw − b_{n−1})·mean(q̃_1 … q̃_{n−Nw−1})."""
    def __init__(self, alpha, Tw, tail=None, h=T_C):
        self.alpha, self.h, self.tail = alpha, h, tail; self.Nw = max(1, int(round(Tw/h))); self.c = h**(-alpha)/Gamma(2 - alpha)
        self.bw = l1_weights(alpha, self.Nw + 1)
    def reset(self, n_max):
        Nw = self.Nw; self.buf = np.zeros((2*Nw, 3)); self.qb = np.zeros((Nw + 1, 3)); self.n = 0; self.q0 = None; self.q_last = None
        self.ssum = np.zeros(3); self.bk = l1_weights(self.alpha, n_max + 2) if self.tail == "mean" else None
        self.brev = self.bw[:Nw][::-1].copy()
    def push(self, q):
        if self.q0 is None: self.q0 = q.copy(); self.q_last = q.copy(); self.qb[0] = 0.0; return np.zeros(3)
        self.n += 1; n, Nw = self.n, self.Nw; d = q - self.q_last; self.q_last = q.copy()
        i = (n - 1) % Nw; self.buf[i] = d; self.buf[i + Nw] = d          # contiguous view of the last Nw increments
        m = min(n, Nw); D = self.brev[Nw - m:] @ self.buf[i + Nw - m + 1:i + Nw + 1]
        self.qb[n % (Nw + 1)] = q - self.q0                              # q̃ of the last Nw + 1 samples
        if self.tail and n > Nw:
            D = D + self.bw[Nw]*self.qb[(n - Nw) % (Nw + 1)]              # leading term b_Nw·q̃_{n−Nw}
            if self.tail == "mean" and n - Nw - 1 >= 1:
                self.ssum += self.qb[(n - Nw - 1) % (Nw + 1)]            # running sum of q̃_1 … q̃_{n−Nw−1}
                D = D - (self.bw[Nw] - self.bk[n - 1])*self.ssum/(n - Nw - 1)
        return self.c * D

class MemWindow(MemWindowNB06):
    """L1 sum over the last N_w increments + O(1) tail correction of Proposition 1 (corrected slot order)."""
    def push(self, q):
        if self.q0 is None: self.q0 = q.copy(); self.q_last = q.copy(); self.qb[0] = 0.0; return np.zeros(3)
        self.n += 1; n, Nw = self.n, self.Nw; d = q - self.q_last; self.q_last = q.copy()
        i = (n - 1) % Nw; self.buf[i] = d; self.buf[i + Nw] = d          # contiguous view of the last Nw increments
        m = min(n, Nw); D = np.einsum("i,ij->j", self.brev[Nw - m:], self.buf[i + Nw - m + 1:i + Nw + 1])
        if self.tail and n > Nw:
            D = D + self.bw[Nw]*self.qb[(n - Nw) % (Nw + 1)]              # leading term b_Nw·q̃_{n−Nw}
            if self.tail == "mean" and n - Nw - 1 >= 1:
                self.ssum += self.qb[(n - Nw - 1) % (Nw + 1)]            # q̃_{n−Nw−1}: read before its slot is overwritten
                D = D - (self.bw[Nw] - self.bk[n - 1])*self.ssum/(n - Nw - 1)
        self.qb[n % (Nw + 1)] = q - self.q0                              # q̃_n (overwrites q̃_{n−Nw−1})
        return self.c * D

class MemOustaloup:
    """Oustaloup approximation of s^α on [ω_b, ω_h] with n = 2N+1 modes, K·(1 + Σ r_k/(s + p_k)),
    each mode Tustin-discretized at h; input q̃ = q − q_0 (Caputo: zero initial state)."""
    def __init__(self, alpha, n, wb, wh, h=T_C):
        N = (n - 1)//2; k = np.arange(-N, N + 1)
        self.p = wb*(wh/wb)**((k + N + (1 + alpha)/2)/(2*N + 1)); z = wb*(wh/wb)**((k + N + (1 - alpha)/2)/(2*N + 1)); self.K = wh**alpha
        self.r = np.array([np.prod(z - pk)/np.prod(np.delete(self.p, i) - pk) for i, pk in enumerate(self.p)])
        self.A = ((2 - self.p*h)/(2 + self.p*h))[:, None]; self.Bd = (h/(2 + self.p*h))[:, None]
    def reset(self, n_max): self.X = np.zeros((len(self.p), 3)); self.u_prev = np.zeros(3); self.q0 = None
    def push(self, q):
        if self.q0 is None: self.q0 = q.copy()
        u = q - self.q0; self.X = self.A*self.X + self.Bd*(u + self.u_prev); self.u_prev = u
        return self.K*(u + self.r @ self.X)

def make_mem(kind, alpha, **kw):
    return {"none": MemNone, "l1": MemL1, "win": MemWindow, "oust": MemOustaloup}[kind](alpha, **kw)

# ---- checks -------------------------------------------------------------------------------------
_t = np.linspace(0, 6, 3001); _q = np.column_stack([np.sin(_t), 0.3*np.cos(2*_t), _t/6])
def online(mem, x): mem.reset(len(x)); return np.array([mem.push(v) for v in x])
_ref = caputo_l1(_q, T_C, 0.5)
print("L1 direct vs batch: max diff %.1e" % np.abs(online(MemL1(0.5), _q) - _ref).max())
print("window covering the whole run vs L1: max diff %.1e" % np.abs(online(MemWindow(0.5, 10.0), _q) - _ref).max())
_pl = PlantL1(0.5, T_C, len(_q), 3, L=400); print("block-FFT plant L1 vs batch: max diff %.1e" % np.abs(np.array([_pl.push(v) for v in _q]) - _ref).max())
_w = np.logspace(-3, 3.5, 300); _o = MemOustaloup(0.5, 7, 1e-2, 1e3)
_H = _o.K*(1 + (_o.r[None, :]/(1j*_w[:, None] + _o.p[None, :])).sum(1))
_band = (_w > 1e-1) & (_w < 1e2)
print("Oustaloup 7 [1e-2, 1e3], α = 0.5, continuous-time: max |gain error| %.2f dB, max phase error %.1f deg on [0.1, 100] rad/s"
      % (np.abs(20*np.log10(np.abs(_H[_band])/_w[_band]**0.5)).max(), np.abs(np.degrees(np.angle(_H[_band])) - 45).max()))

## New realizations: sum of exponentials (SOE) and the exact fast L1 (block FFT)

**SOE.** The L1 sum is `D_n = c·Σ_{k=1..n} b_{n−k}·Δq_k`, with `c·b_j = (1/h)∫_{t_{j}}^{t_{j+1}} K(τ) dτ` and `K(τ) = τ^{−α}/Γ(1−α)`. The last increment (`j = 0`) is kept exact. For the history (`τ ≥ h`) the kernel is replaced by `K(τ) ≈ Σ_i w_i e^{−λ_i τ}`, with relative error ≤ ε on `[h, T_hor]`, from the trapezoidal rule in `x = ln s` on `τ^{−α} = (1/Γ(α))∫ exp(−τ e^x + αx) dx` (nodes `λ_i = e^{x_i}`; the step and the truncation are chosen so that the measured error on a fine grid of `[h, T_hor]` is below ε). Each exponential then gives one recursive state per joint, `S_i(n) = e^{−λ_i h}·S_i(n−1) + g_i·Δq_{n−1}` with `g_i = e^{−λ_i h}(1 − e^{−λ_i h})/(λ_i h)` (exact integral over the step), and `D_n = c·Δq_n + Σ_i w_i S_i(n)`. Up to `T_hor` the error is bounded by `ε·c·Σ_{j≥1} b_j |Δq_{n−j}|` (checked in E1); beyond `T_hor` the SOE forgets faster than `τ^{−α}`. Cost: about 4 flops per mode and joint, `n_SOE` states per joint, constant in time.

**Fast L1 (block FFT).** The same L1 sum, exact, with the history older than the current block of `L` steps convolved by one FFT at the start of each block and the recent samples summed directly. Mean cost per step ≈ `FFT(n)/L + 6L` flops, but the step that computes the FFT costs `O(n log n)` (worst case), and the stored history grows as `n`.

In [ ]:
_SOE_CACHE = {}
def soe_nodes(alpha, h, T, eps):
    """λ_i, w_i with |Σ w_i e^{−λ_i τ} / (τ^{−α}/Γ(1−α)) − 1| ≤ eps on [h, T] (max over 3000 log-spaced τ)."""
    key = (round(alpha, 6), h, T, eps)
    if key in _SOE_CACHE: return _SOE_CACHE[key]
    tau = np.logspace(np.log10(h), np.log10(T), 3000); Kt = tau**(-alpha)/Gamma(1 - alpha)
    for safety in (1.0, 0.3, 0.1, 0.03):              # tighten the truncation if needed
        xmin = np.log(safety*eps*alpha*Gamma(alpha))/alpha - np.log(T); xmax = np.log((np.log(1/(safety*eps)) + 5.0)/h)
        for hx in np.arange(2.0, 0.05, -0.01):
            x = np.arange(xmin, xmax + hx, hx); lam = np.exp(x); w = hx*np.exp(alpha*x)/(Gamma(alpha)*Gamma(1 - alpha))
            keep = w*np.exp(-lam*h) > 1e-300; lam, w = lam[keep], w[keep]
            err = np.abs(np.exp(-np.outer(tau, lam)) @ w / Kt - 1).max()
            if err <= eps: _SOE_CACHE[key] = (lam, w, err); return lam, w, err
    raise RuntimeError("SOE construction failed")

class MemSOE:
    """L1 sum with the history kernel replaced by a sum of exponentials (exact last step)."""
    def __init__(self, alpha, eps, T, h=T_C):
        self.alpha, self.h = alpha, h; self.c = h**(-alpha)/Gamma(2 - alpha)
        lam, w, self.kerr = soe_nodes(alpha, h, T, eps); E = np.exp(-lam*h)
        self.E = E[:, None]; self.G = (E*(1 - E)/(lam*h))[:, None]; self.w = w.copy(); self.nm = len(lam)
    def reset(self, n_max): self.S = np.zeros((self.nm, 3)); self.d_prev = np.zeros(3); self.q_last = None
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros(3)
        d = q - self.q_last; self.q_last = q.copy()
        self.S = self.E*self.S + self.G*self.d_prev; self.d_prev = d
        return self.c*d + np.einsum("i,ij->j", self.w, self.S)

class MemL1Block:
    """Exact full-memory L1 with block FFT (old history once per block of L steps, recent samples directly)."""
    def __init__(self, alpha, L=1024, h=T_C): self.alpha, self.h, self.L = alpha, h, L; self.c = h**(-alpha)/Gamma(2 - alpha)
    def reset(self, n_max):
        L = self.L; self.b = l1_weights(self.alpha, n_max + L + 2); self.br = self.b[:L][::-1].copy()
        self.dq = np.zeros((n_max + 2, 3)); self.n = 0; self.n0 = 1; self.q_last = None; self.far = np.zeros((L, 3))
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros(3)
        self.n += 1; n = self.n; self.dq[n] = q - self.q_last; self.q_last = q.copy()
        if (n - 1) % self.L == 0:
            self.n0 = n
            self.far = (fftconvolve(self.dq[1:n], self.b[:n - 1 + self.L][:, None], axes=0)[n - 1:n - 1 + self.L] if n > 1 else np.zeros((self.L, 3)))
        j = n - self.n0
        return self.c*(self.far[j] + np.einsum("i,ij->j", self.br[self.L - j - 1:], self.dq[self.n0:n + 1]))

def make_mem(kind, alpha, **kw):
    return {"none": MemNone, "l1": MemL1, "win": MemWindow, "oust": MemOustaloup, "soe": MemSOE, "l1block": MemL1Block}[kind](alpha, **kw)

# ---- vectorized (offline) versions of the same realizations, for the 60 min references --------------
def off_eval(kind, alpha, x, h=T_C, **kw):
    """D^α of x (N, 3) sampled every h from t = 0, by realization `kind`; same arithmetic as push() up to rounding."""
    N = len(x); c = h**(-alpha)/Gamma(2 - alpha); d = np.vstack([np.zeros((1, 3)), np.diff(x, axis=0)]); xt = x - x[0]
    if kind == "none": return np.zeros_like(x)
    if kind == "l1": return caputo_l1(x, h, alpha)
    if kind == "win":
        Nw = max(1, int(round(kw["Tw"]/h))); b = l1_weights(alpha, N + 1)
        D = fftconvolve(d, b[:Nw][:, None], axes=0)[:N]
        if kw.get("tail"):
            n = np.arange(N); D[Nw + 1:] += b[Nw]*xt[1:N - Nw]                                  # b_Nw·q̃_{n−Nw}, n > Nw
            if kw["tail"] == "mean":
                cs = np.cumsum(xt, axis=0); m = n - Nw - 1; ok = m >= 1                        # mean of q̃_1 … q̃_{n−Nw−1}
                mean = np.zeros_like(x); mean[ok] = (cs[m[ok]] - xt[0]) / m[ok][:, None]
                D[ok] -= (b[Nw] - b[n[ok] - 1])[:, None]*mean[ok]
        return c*D
    if kind == "oust":
        o = MemOustaloup(alpha, kw["n"], kw["wb"], kw["wh"], h); v = xt + np.vstack([np.zeros((1, 3)), xt[:-1]]); out = xt.copy()
        for k in range(len(o.p)): out += o.r[k]*lfilter([o.Bd[k, 0]], [1.0, -o.A[k, 0]], v, axis=0)
        return o.K*out
    if kind == "soe":
        s = MemSOE(alpha, kw["eps"], kw["T"], h); dp = np.vstack([np.zeros((1, 3)), d[:-1]]); out = c*d
        for i in range(s.nm): out += s.w[i]*lfilter([s.G[i, 0]], [1.0, -s.E[i, 0]], dp, axis=0)
        return out
    raise ValueError(kind)

# ---- checks -------------------------------------------------------------------------------------
_ref = caputo_l1(_q, T_C, 0.5)
for _kind, _kw in [("soe", dict(eps=1e-3, T=60.0)), ("l1block", dict(L=256)), ("win", dict(Tw=0.5, tail=None)), ("win", dict(Tw=0.5, tail="lead")), ("win", dict(Tw=0.5, tail="mean")), ("oust", dict(n=7, wb=1e-2, wh=1e3))]:
    _on = online(make_mem(_kind, 0.5, **_kw), _q)
    _off = off_eval(_kind if _kind != "l1block" else "l1", 0.5, _q, **{k: v for k, v in _kw.items() if k != "L"})
    print(f"{_kind:8s} {str(_kw):40s}: online vs vectorized max diff {np.abs(_on - _off).max():.1e} | rel. RMS diff vs L1 {np.sqrt(np.mean((_on - _ref)**2)/np.mean(_ref**2)):.1e}")
for _a in ALPHAS_OFF:
    print(f"SOE nodes α = {_a}: " + ", ".join(f"ε {fmt_eps(e)} -> {len(soe_nodes(_a, T_C, SOE_HOR, e)[0])}" for e in SOE_EPS) + f" (horizon {fmt_hor(SOE_HOR)})")

## References of the closed loop: one Adept cycle and continuous operation (NB06)

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05

def adept_P(scale):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]; P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    return P

def reference(scale, cycles=1):
    P1 = adept_P(scale); P = np.vstack([P1] + [P1[1:]]*(cycles - 1)); n = len(P)
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(t=np.arange(n)*dt, P=P, q=q, qd=qd, qdd=qdd, N=n, n_cycle=len(P1) - 1)

SCEN = {f"Adept {sc:g} s": (sc, 1) for sc in SINGLE_SCALES}; SCEN[f"Continuous {N_CYCLES}x{CONT_SCALE:g} s"] = (CONT_SCALE, N_CYCLES)
REFS = {k: reference(*v) for k, v in SCEN.items()}
for k, R in REFS.items(): print(f"closed loop | {k:22s}: {R['t'][-1]:5.1f} s, {R['N']} steps")

## E1 references: 60 min of operation with pauses and work away from the initial pose (controller rate, 500 Hz)

Built from the Adept cycle of NB06 (which starts and ends at the ready pose `q(0)`) and the pick pose `P1 = (0.10, 0.10, 0.55) m`, its first waypoint, where the joints are displaced by `q̃ = (0.07, 0.25, 0.64) rad` from `q(0)`:

| Reference | Pattern (repeated until 60 min) | What it tests |
|---|---|---|
| `Periodic` | Adept 1.6 s cycles back to back (the 60 s continuous run of NB06, extended to 60 min) | zero-mean periodic motion: only the start transient is low-frequency |
| `Pauses at ready` | 5 min of 1.6 s cycles, 2 min at rest at the ready pose, 3 min of 3.2 s cycles | rests where `q̃ = 0` |
| `Pauses at pick pose` | 4 min of 1.6 s cycles, move to `P1` (0.4 s), **hold 6 min**, back to ready | a displacement held for several `1/ω_b` (= 100 s for ω_b = 10⁻² rad/s): the low-frequency limit of Oustaloup |
| `Away from ready` | move to `P1`, then the square `P1 → P2 → P3 → P4 → P1` of the Adept cycle without returning to ready (3.2 s per square), with a 1 min hold at `P1` every 10 min | non-zero mean displacement during the whole hour |

In [ ]:
def quintic(P0, P1, T, fs, dip=0.0):
    n = max(2, int(round(T*fs))); tau = np.arange(1, n + 1)/n; s = 6*tau**5 - 15*tau**4 + 10*tau**3
    P = P0 + (P1 - P0)*s[:, None]; P[:, 2] -= dip*np.sin(np.pi*tau)**2; return P
def hold(P0, T, fs): return np.repeat(P0[None], max(1, int(round(T*fs))), 0)
def adept_cycle(scale, fs):
    segs = [quintic(WAYPOINTS[i], WAYPOINTS[i + 1], (BASE_T[i + 1] - BASE_T[i])*scale, fs, H_P) for i in range(len(WAYPOINTS) - 1)]
    return np.vstack(segs)
def cycles_for(T, scale, fs):
    c = adept_cycle(scale, fs); k = max(1, int(round(T/(len(c)/fs)))); return np.vstack([c]*k)
P1, P_SQ = WAYPOINTS[1], WAYPOINTS[1:6]
def square(scale, fs): return np.vstack([quintic(P_SQ[i], P_SQ[i + 1], (BASE_T[i + 2] - BASE_T[i + 1])*scale, fs, H_P) for i in range(4)])

def long_reference(name, total_s, fs=1.0/T_C):
    k = total_s/3600.0                                   # QUICK: all durations scaled by total/60 min (holds at least 10 s)
    parts = [P_READY[None]]
    while sum(len(p) for p in parts) < total_s*fs:
        if name == "Periodic": parts.append(adept_cycle(1.6, fs))
        elif name == "Pauses at ready":
            parts += [cycles_for(300*k, 1.6, fs), hold(P_READY, max(10, 120*k), fs), cycles_for(180*k, 3.2, fs)]
        elif name == "Pauses at pick pose":
            parts += [cycles_for(240*k, 1.6, fs), quintic(P_READY, P1, 0.4, fs), hold(P1, max(10, 360*k), fs), quintic(P1, P_READY, 0.4, fs)]
        elif name == "Away from ready":
            if len(parts) == 1: parts.append(quintic(P_READY, P1, 0.4, fs))
            parts += [np.vstack([square(1.6, fs)]*max(1, int(round(540*k/3.2)))), hold(P1, max(10, 60*k), fs)]
    P = np.vstack(parts)[:int(round(total_s*fs)) + 1]; q = ik_vec(P)
    assert np.all(np.isfinite(q)), name
    return dict(t=np.arange(len(P))/fs, P=P, q=q, N=len(P))
LONG_NAMES = ("Periodic", "Pauses at ready", "Pauses at pick pose", "Away from ready")
t0 = time.time(); LREFS = {nm: long_reference(nm, 60.0*LONG_MIN) for nm in LONG_NAMES}
for nm, R in LREFS.items():
    qt = R["q"] - R["q"][0]
    print(f"{nm:20s}: {R['t'][-1]/60:5.1f} min, {R['N']} samples | mean |q̃| {np.abs(qt).mean():.3f} rad, max |q̃| {np.abs(qt).max():.3f} rad")
print(f"({time.time() - t0:.0f} s)")

## Correction of the NB06 window + tail code

In the first NB06 run (repository v1.0 of this study; the NB06 of release v1.1.0 already contains the fix) `MemWindow.push` wrote `q̃_n` into the ring buffer of the last `N_w + 1` samples **before** reading `q̃_{n−N_w−1}` from it, and both live in the same slot (`n ≡ n − N_w − 1 mod N_w + 1`). The running sum therefore accumulated the newest sample instead of the oldest discarded one, so the "mean" correction used the mean of `q̃` over the samples `N_w + 2 … n` (the recent past) instead of `1 … n − N_w − 1` (the discarded history) as Proposition 1 states. The leading term `b_Nw·q̃_{n−N_w}` and the plain windows were correct, and so are Oustaloup and the L1 sum. The same order was in the C kernel. Below, `MemWindow` reads the slot before overwriting it; `MemWindowNB06` keeps the old code, and E0 measures the difference on the NB06 references. **All `+ tail` results of NB06 (offline table and closed loop) are recomputed here with the corrected code.**

In [ ]:
NB06_OFF_CSV = "NB06_fPINN_ErrorMemoria.csv"
e0 = []; t0 = time.time()
for alpha in ALPHAS_OFF:
    for scen, R in REFS.items():
        qc = R["q"][::STEP_C]; ref = caputo_l1(qc, T_C, alpha); s = np.sqrt(np.mean(ref**2))
        for tw in ((0.05, 0.1, 0.2, 0.5, 1.0, 2.0) if not QUICK else (0.1, 0.5)):
            old = online(MemWindowNB06(alpha, tw, "mean"), qc); new = off_eval("win", alpha, qc, Tw=tw, tail="mean")
            e0.append({"alpha": alpha, "Scenario": scen, "Realization": f"window {tw:g} s + tail",
                       "Rel. RMS error, NB06 code (%)": 100*np.sqrt(np.mean((old - ref)**2))/s, "Rel. RMS error, corrected (%)": 100*np.sqrt(np.mean((new - ref)**2))/s,
                       "Rel. RMS error, plain window (%)": 100*np.sqrt(np.mean((off_eval("win", alpha, qc, Tw=tw) - ref)**2))/s})
e0 = pd.DataFrame(e0); e0.to_csv(f"NB07_fPINN_CorreccionCola{SUFFIX}.csv", index=False)
if os.path.exists(NB06_OFF_CSV):
    o = pd.read_csv(NB06_OFF_CSV).rename(columns={"Rel. RMS error (%)": "NB06 csv (%)"})
    m = e0.merge(o[["alpha", "Scenario", "Realization", "NB06 csv (%)"]], on=["alpha", "Scenario", "Realization"])
    if len(m): print(f"NB06 code here vs NB06_fPINN_ErrorMemoria.csv ({len(m)} rows): max abs diff {np.abs(m['Rel. RMS error, NB06 code (%)'] - m['NB06 csv (%)']).max():.1e} %")
print(f"({time.time() - t0:.0f} s) relative RMS error of D^α q (%), NB06 code -> corrected, α = {ALPHA_MAIN}:")
print(e0[e0.alpha == ALPHA_MAIN].round(2).to_string(index=False))

## E1: offline error of the memory torque over 60 min (no networks, no loop)

Memory torque `B·D^α q_ref` of every realization against the full L1 sum at `T_C` (the operator the controller of NB03/NB04/NB06 uses), with `B = 0.05 N m s^α`. Per minute: RMS and maximum torque error (mN m) and RMS error relative to the RMS of the L1 torque in that minute. Window lengths, Oustaloup bands and SOE horizons are those of NB06 plus wider ones (`[10⁻⁴, 10³]`, `T_hor = 1 h`) that answer the low-frequency question. For the SOE, the column `SOE bound ratio` is `max |e| / (ε·c·Σ_{j≥1} b_j|Δq_{n−j}|)` over `t ≤ T_hor` (≤ 1 confirms the a priori bound; samples where the bound is at rounding level, during rests, are skipped, and `SOE bound excess` gives the largest excess in mN m over all samples).

In [ ]:
OFF_LONG = ([("none", "none", {})]
            + [(f"window {tw:g} s" + (" + tail" if tl else ""), "win", dict(Tw=tw, tail=tl)) for tw, tl in ((0.5, None), (2.0, None), (0.5, "mean"), (2.0, "mean"), (10.0, "mean"))]
            + [(f"Oustaloup {n} [{fmt_eps(wb)}, 1e3]", "oust", dict(n=n, wb=wb, wh=1e3)) for n, wb in ((5, 1e-2), (7, 1e-2), (9, 1e-2), (7, 1e-3), (9, 1e-4), (11, 1e-4))]
            + [(f"SOE {fmt_eps(e)} [{fmt_hor(T)}]", "soe", dict(eps=e, T=T)) for e, T in ((1e-2, SOE_HOR), (1e-3, SOE_HOR), (1e-4, SOE_HOR), (1e-3, 60.0))])
if QUICK: OFF_LONG = [r for r in OFF_LONG if r[0] in ("none", "window 0.5 s + tail", "Oustaloup 7 [1e-2, 1e3]", "Oustaloup 9 [1e-4, 1e3]", "SOE 1e-3 [1 h]", "SOE 1e-3 [60 s]")]
BIN_S = 60.0 if not QUICK else 6.0                     # length of a "minute" bin [s]
def cost(kind, kw, alpha=0.5, n=0):
    """flops per update and RAM [B] for 3 joints (double); n = stored history for the L1 sum."""
    if kind == "l1":   return 6*n, 8*4*n
    if kind == "l1block": L = kw["L"]; return None, 8*(4*n + 3*L + L)
    if kind == "win":  Nw = int(round(kw["Tw"]/T_C)); return 6*Nw + (18 if kw.get("tail") else 0), 8*(2*3*Nw + 3*(Nw + 1) + (Nw + 1))
    if kind == "oust": return 3*(4*kw["n"] + 2), 8*(3*kw["n"] + 3*kw["n"] + 6)
    if kind == "soe":  m = len(soe_nodes(alpha, T_C, kw["T"], kw["eps"])[0]); return 3*(5*m + 2), 8*(3*m + 3*m + 6)
    return 0, 0

OUT_MIN, OUT_RES = f"NB07_fPINN_LargoPorMinuto{SUFFIX}.csv", f"NB07_fPINN_LargoResumen{SUFFIX}.csv"
rows_min = pd.read_csv(OUT_MIN).to_dict("records") if os.path.exists(OUT_MIN) else []
rows_res = pd.read_csv(OUT_RES).to_dict("records") if os.path.exists(OUT_RES) else []
done_off = {(r["alpha"], r["Reference"], r["Realization"]) for r in rows_res}
t0 = time.time()
for alpha in ALPHAS_OFF:
    for nm, R in LREFS.items():
        todo = [r for r in OFF_LONG if (alpha, nm, r[0]) not in done_off]
        if not todo: continue
        x = R["q"]; N = len(x); ref = caputo_l1(x, T_C, alpha); nb = int(round(BIN_S/T_C)); nbin = int(np.ceil(N/nb))
        absd = np.abs(np.vstack([np.zeros((1, 3)), np.diff(x, axis=0)])); bw = l1_weights(alpha, N + 1); bw[0] = 0.0
        hist_abs = (T_C**(-alpha)/Gamma(2 - alpha))*fftconvolve(absd, bw[:N][:, None], axes=0)[:N]   # c·Σ_{j≥1} b_j|Δq_{n−j}|
        for label, kind, kw in todo:
            e = B_FRAC*(off_eval(kind, alpha, x, **kw) - ref)*1e3; r_ = B_FRAC*ref*1e3          # mN m
            ae = np.abs(e).max(1)
            for b in range(nbin):
                sl = slice(b*nb, min(N, (b + 1)*nb))
                rows_min.append({"alpha": alpha, "Reference": nm, "Realization": label, "Bin start (min)": b*BIN_S/60,
                                 "Memory torque RMS (mN m)": np.sqrt(np.mean(r_[sl]**2)), "Error RMS (mN m)": np.sqrt(np.mean(e[sl]**2)),
                                 "Error max (mN m)": ae[sl].max(), "Rel. RMS error (%)": 100*np.sqrt(np.mean(e[sl]**2)/max(np.mean(r_[sl]**2), 1e-30))})
            n60, n10 = int(60/T_C), int(min(600, 0.2*R["t"][-1])/T_C)
            fl, by = cost(kind, kw, alpha)
            row = {"alpha": alpha, "Reference": nm, "Realization": label, "flops per update": fl, "RAM (kB)": by/1024,
                   "Rel. RMS error, whole run (%)": 100*np.sqrt(np.mean(e**2)/np.mean(r_**2)),
                   "Rel. RMS error, first 60 s (%)": 100*np.sqrt(np.mean(e[:n60]**2)/np.mean(r_[:n60]**2)),
                   "Rel. RMS error, last 10 min (%)": 100*np.sqrt(np.mean(e[-n10:]**2)/np.mean(r_[-n10:]**2)),
                   "Error RMS, whole run (mN m)": np.sqrt(np.mean(e**2)), "Error max, whole run (mN m)": ae.max(),
                   "Error max, after 10 min (mN m)": ae[int(min(600, 0.2*R["t"][-1])/T_C):].max(), "Memory torque RMS (mN m)": np.sqrt(np.mean(r_**2))}
            if kind == "soe":
                m = R["t"] <= kw["T"]; bound = 1e3*B_FRAC*kw["eps"]*hist_abs[m]; em = np.abs(e[m]); big = bound > 1e-3*bound.max()   # skip rounding-level samples (rests)
                row["SOE bound ratio (t <= T_hor)"] = (em[big]/bound[big]).max(); row["SOE bound excess, max (mN m)"] = max(0.0, (em - bound).max())
                row["SOE modes"] = len(soe_nodes(alpha, T_C, kw["T"], kw["eps"])[0])
            rows_res.append(row)
        pd.DataFrame(rows_min).to_csv(OUT_MIN, index=False); pd.DataFrame(rows_res).to_csv(OUT_RES, index=False)
        print(f"α {alpha} | {nm:20s} | {len(todo)} realizations | elapsed {time.time() - t0:.0f} s", flush=True)
lmin, lres = pd.DataFrame(rows_min), pd.DataFrame(rows_res)
print("\nrelative RMS error over the whole run (%), α = 0.5:")
print(lres[lres.alpha == 0.5].pivot(index="Realization", columns="Reference", values="Rel. RMS error, whole run (%)").round(3).to_string())
print("\nmax memory torque error after 10 min (mN m), α = 0.5:")
print(lres[lres.alpha == 0.5].pivot(index="Realization", columns="Reference", values="Error max, after 10 min (mN m)").round(3).to_string())

## E1b: analytic low-frequency limit of Oustaloup and step error, checked against the holds

Oustaloup in modal form is `H(s) = K(1 + Σ r_k/(s + p_k)) = K Π (s + z_k)/(s + p_k)` with `K = ω_h^α` and `z_k/p_k = (ω_h/ω_b)^{−α/n}`, so **`H(0) = ω_b^α` exactly** (Tustin keeps the DC gain). For a displacement `q̃` applied at `t_0` and held, the true Caputo derivative is `q̃·τ^{−α}/Γ(1−α) → 0` (`τ = t − t_0`), while the filter output tends to `ω_b^α q̃`. The memory-torque error of the step is

`e_step(τ) = B q̃ [K(1 + Σ_k (r_k/p_k)(1 − e^{−p_k τ})) − τ^{−α}/Γ(1−α)]`, with limit **`B ω_b^α q̃`** as `τ → ∞`,

reached after a few `1/ω_b`. With α = 0.5, ω_b = 10⁻² rad/s and `q̃ = 0.64 rad` it is 3.2 mN m. For the window + tail the error of a held displacement is zero (summation by parts is exact for a constant `q̃` over the discarded part), and for the SOE it is bounded by ε times the L1 history up to `T_hor`. The table gives `e_step` per unit displacement at several hold times and the limit for the largest joint displacement of the pick pose, and compares it with the error measured at the end of each 6 min hold of `Pauses at pick pose` (joint 3, the most displaced one; the remaining difference is the memory of the preceding motion).

In [ ]:
QT_PICK = (ik_vec(P1[None]) - ik_vec(P_READY[None]))[0]; j3 = int(np.argmax(np.abs(QT_PICK)))
def oust_step_err(alpha, n, wb, wh, tau):
    o = MemOustaloup(alpha, n, wb, wh); return o.K*(1 + ((o.r/o.p)[None, :]*(1 - np.exp(-np.outer(tau, o.p)))).sum(1)) - tau**(-alpha)/Gamma(1 - alpha)
TAUS = np.array([1, 10, 100, 360, 600, 3600, 1e5])
cota = []; Rp = LREFS["Pauses at pick pose"]; qt = Rp["q"] - Rp["q"][0]
# ends of the holds at the pick pose: last sample of each run of constant q
still = np.r_[False, np.all(np.abs(np.diff(Rp["q"], axis=0)) < 1e-12, 1)]; ends = np.where(still[:-1] & ~still[1:])[0]
ends = [i for i in ends if abs(qt[i, j3]) > 0.5]
for alpha in ALPHAS_OFF:
    ref = caputo_l1(Rp["q"], T_C, alpha)
    for label, kind, kw in [r for r in OFF_LONG if r[1] in ("oust", "win", "soe") and (r[1] != "win" or r[2].get("tail"))]:
        row = {"alpha": alpha, "Realization": label}
        if kind == "oust":
            o = MemOustaloup(alpha, kw["n"], kw["wb"], kw["wh"])
            row.update({"H(0)": o.K*(1 + (o.r/o.p).sum()), "wb^alpha": kw["wb"]**alpha,
                        "Limit B*wb^a*|q~| (mN m)": 1e3*B_FRAC*kw["wb"]**alpha*abs(QT_PICK[j3])})
            es = oust_step_err(alpha, kw["n"], kw["wb"], kw["wh"], TAUS)
            row.update({f"e_step/q~ at {t:g} s (mN m/rad)": 1e3*B_FRAC*v for t, v in zip(TAUS, es)})
            row["Predicted error at end of hold (mN m)"] = 1e3*B_FRAC*abs(QT_PICK[j3])*abs(oust_step_err(alpha, kw["n"], kw["wb"], kw["wh"], np.array([max(10, 360*LONG_MIN/60)]))[0])
        if ends:
            e = 1e3*B_FRAC*(off_eval(kind, alpha, Rp["q"], **kw) - ref)[ends, j3]
            row["Measured |error| at end of holds, mean (mN m)"] = np.abs(e).mean(); row["Measured |error| at end of holds, max (mN m)"] = np.abs(e).max()
            row["L1 memory torque at end of holds, mean (mN m)"] = np.abs(1e3*B_FRAC*ref[ends, j3]).mean()
        cota.append(row)
cota = pd.DataFrame(cota); cota.to_csv(f"NB07_fPINN_CotaOustaloup{SUFFIX}.csv", index=False)
print(f"pick pose q̃ = {np.round(QT_PICK, 3)} rad (joint {j3 + 1} used), {len(ends)} holds")
print(cota[cota.alpha == 0.5].drop(columns=[c for c in cota if c.startswith("e_step")]).round(4).to_string(index=False))

## E2a: SOE size and kernel accuracy, and Oustaloup read as a sum of exponentials

Oustaloup in modal form is itself a sum of exponentials: `H(s) = s·G(s)` with `G(s) = H(s)/s`, so `D^α q = ∫ g(τ) q̇(t − τ) dτ` with
`g(τ) = ω_b^α − Σ_k K (r_k/p_k) e^{−p_k τ}`, i.e. **a constant term (a pole at s = 0, which never forgets) plus exponentials whose poles are fixed by the frequency band**, while the SOE places its poles from the time-domain kernel and has no constant term. The table gives, for each α, the number of modes and the maximum relative error of the kernel `g(τ)` against `τ^{−α}/Γ(1−α)` on `[T_C, 60 s]`, `[T_C, 10 min]` and `[T_C, 1 h]` for both families.

In [ ]:
def kernel_err(lam, w, alpha, T, const=0.0):
    tau = np.logspace(np.log10(T_C), np.log10(T), 3000); Kt = tau**(-alpha)/Gamma(1 - alpha)
    return np.abs((const + np.exp(-np.outer(tau, lam)) @ w)/Kt - 1).max()
soe_rows = []
for alpha in sorted(set(ALPHAS_OFF) | {ALPHA_MAIN} | set(ALPHA_SWEEP)):
    for label, kind, kw in [r for r in OFF_LONG if r[1] in ("soe", "oust")]:
        if kind == "soe": lam, w, _ = soe_nodes(alpha, T_C, kw["T"], kw["eps"]); const = 0.0
        else: o = MemOustaloup(alpha, kw["n"], kw["wb"], kw["wh"]); lam, w, const = o.p, -o.K*o.r/o.p, kw["wb"]**alpha
        fl, by = cost(kind, kw, alpha)
        soe_rows.append({"alpha": alpha, "Realization": label, "Family": kind, "modes (per joint)": len(lam), "flops per update": fl, "RAM (kB)": by/1024,
                         "slowest pole (rad/s)": lam.min(), "fastest pole (rad/s)": lam.max(),
                         **{f"kernel max rel. error on [T_C, {fmt_hor(T)}]": kernel_err(lam, w, alpha, T, const) for T in (60.0, 600.0, 3600.0)}})
soe_tab = pd.DataFrame(soe_rows); soe_tab.to_csv(f"NB07_fPINN_SOE{SUFFIX}.csv", index=False)
print(soe_tab[soe_tab.alpha == ALPHA_MAIN].round(4).to_string(index=False))

## Models: fPINN and PINN networks of NB02/NB04 (175 trajectories), and the fractional oracle

In [ ]:
def silu(z): return z / (1.0 + np.exp(-z))
class NumpyFF:
    """Forward pass of NB02/NB04 for a batch (kinds MLP, HYBRID, PINN, fHYBRID, fPINN)."""
    def __init__(self, path):
        d = np.load(path, allow_pickle=False); self.kind = str(d["kind"]); self.core = {"fPINN": "PINN", "fHYBRID": "HYBRID"}.get(self.kind, self.kind)
        self.W = [d[k].astype(float) for k in ("W1", "b1", "W2", "b2", "W3", "b3")]
        self.x_mean, self.x_std = d["x_mean"].astype(float), d["x_std"].astype(float)
        self.y_mean, self.y_std, self.r_std = d["y_mean"].astype(float), d["y_std"].astype(float), d["r_std"].astype(float)
        self.alpha, self.B = float(d["alpha"]), d["B"].astype(float)
    def F(self, q, qd, qdd, m, p):
        W1, b1, W2, b2, W3, b3 = self.W
        xs = (np.column_stack([q, qd, qdd, np.full(len(q), m), p]) - self.x_mean) / self.x_std
        out = silu(silu(xs @ W1.T + b1) @ W2.T + b2) @ W3.T + b3
        if self.core == "MLP":    return out*self.y_std + self.y_mean
        if self.core == "HYBRID": return gamma_payload(m)*(regressor_b(q, qdd, p) @ THETA0) + out*self.r_std
        return np.einsum("mij,mj->mi", regressor_b(q, qdd, p), out)

def weight_file(kind, alpha, seed):
    scn = "main" if alpha == ALPHA_MAIN else "sweep"
    for f in (os.path.join(NB04_WEIGHTS, f"NB04_fPINN_{scn}_{kind}_{SIZE}traj_seed{seed}_a{alpha}_e{EPOCHS_TAG}.npz"),
              os.path.join(NB02_WEIGHTS, f"NB02_fPINN_{kind}_{SIZE}traj_seed{seed}_a{alpha}_e{EPOCHS_TAG}.npz")):
        if os.path.exists(f): return f
    return None

def model_list():
    """(alpha, label, seed, F, alpha_hat, B_hat, realizations) for every batch."""
    out = []; stand_in = False
    for alpha, seeds in [(ALPHA_MAIN, MAIN_SEEDS)] + [(a, SWEEP_SEEDS) for a in ALPHA_SWEEP]:
        out.append((alpha, "oracle (frac)", 0, F_oracle_b, alpha, np.full(3, B_FRAC), REALIZ))
        for seed in seeds:
            for kind in ("fPINN", "PINN"):
                f = weight_file(kind, alpha, seed)
                if f is None:
                    print(f"missing weights: {kind} {SIZE} traj seed {seed} α {alpha}")
                    if QUICK and kind == "fPINN":   # stand-in so that QUICK runs without weights: oracle dynamics with the typical fPINN bias
                        out.append((alpha, "fPINN (stand-in)", seed, F_oracle_b, alpha - 0.01, np.full(3, 1.1*B_FRAC), REALIZ)); stand_in = True
                    continue
                n = NumpyFF(f)
                out.append((alpha, kind, seed, n.F, n.alpha, n.B, REALIZ if n.B.any() else [("none", "none", {})]))
                if n.B.any(): print(f"α {alpha} {kind} seed {seed}: α̂ = {n.alpha:.3f}, B̂ = {np.round(n.B, 4)}  ({f})")
    if stand_in: print("QUICK without weights: 'fPINN (stand-in)' = oracle dynamics with α̂ = α − 0.01, B̂ = 1.1·B (checks the pipeline only)")
    return out
MODELS = model_list()
print(len(MODELS), "batches")

## Batched closed-loop simulation (CTC + I of NB03 + memory term from each realization)

In [ ]:
KP_PD, KD_PD, KI_RATIO = 150.0, 10.0, 10.0
KP_EQ, KD_EQ = KP_PD/THETA0[0], KD_PD/THETA0[0]

def simulate_batch(F, a_hat, B_hat, realiz, R, alpha, m=PAYLOAD):
    """All realizations of one model on one reference. Returns one metrics dict per realization (None if unstable)."""
    M = len(realiz); N = R["N"]; n_c = N//STEP_C + 2
    mems = [make_mem(kind, a_hat, **kw) for _, kind, kw in realiz]
    for mm in mems: mm.reset(n_c)
    q = np.repeat(R["q"][:1], M, 0); qd = np.repeat(R["qd"][:1], M, 0); I = np.zeros((M, 3)); mem = np.zeros((M, 3))
    plant = PlantL1(alpha, dt, N, 3*M); bad = np.zeros(M, bool)
    E = np.zeros((N, M)); T2 = np.zeros(M); TV = np.zeros(M); Tpk = np.zeros(M); Tprev = None
    MEM = np.zeros((n_c, M, 3)); k_c = 0; nc = R["n_cycle"]
    for i in range(N):
        p = fk_b(q); E[i] = np.linalg.norm(R["P"][i] - p, axis=1); Dq = plant.push(q.reshape(-1)).reshape(M, 3)
        if i % STEP_C == 0:
            for j in range(M): mem[j] = B_hat*mems[j].push(q[j])
            MEM[k_c] = mem; k_c += 1
        e = R["q"][i] - q; ed = R["qd"][i] - qd; I += e*dt
        v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*I
        tau = np.clip(F(q, qd, v, m, p) + mem, -TAU_MAX, TAU_MAX)
        T2 += (tau**2).sum(1); Tpk = np.maximum(Tpk, np.abs(tau).max(1))
        if Tprev is not None: TV += np.linalg.norm(tau - Tprev, axis=1)
        Tprev = tau
        qd = qd + plant_qdd_b(q, qd, tau, p, m, Dq)*dt; q = q + qd*dt
        new_bad = ~np.all(np.isfinite(q), 1) | (np.abs(np.nan_to_num(q, nan=9.0)).max(1) > 3)
        if new_bad.any():                     # freeze a diverged member on the reference so it cannot disturb the others
            bad |= new_bad; q[new_bad] = R["q"][i + 1 if i + 1 < N else i]; qd[new_bad] = 0.0; I[new_bad] = 0.0
    e = 1000*E; MEM = MEM[:k_c]
    i_l1 = next((j for j, r in enumerate(realiz) if r[1] == "l1"), None)
    out = []
    for j, (label, kind, kw) in enumerate(realiz):
        if bad[j]: out.append(None); continue
        d = {"Cartesian RMSE (mm)": np.sqrt(np.mean(e[:, j]**2)), "Max error (mm)": e[:, j].max(),
             "RMSE first cycle (mm)": np.sqrt(np.mean(e[:nc + 1, j]**2)), "RMSE last cycle (mm)": np.sqrt(np.mean(e[-nc - 1:, j]**2)),
             "RMS torque (N m)": np.sqrt(T2[j]/(3*N)), "TV (N m)": TV[j], "Peak |tau| (N m)": Tpk[j],
             "Memory torque RMS (mN m)": 1e3*np.sqrt(np.mean(MEM[:, j]**2))}
        if i_l1 is not None: d["Memory torque diff vs L1 RMS (mN m)"] = 1e3*np.sqrt(np.mean((MEM[:, j] - MEM[:, i_l1])**2))
        out.append(d)
    return out

# check: one batch member reproduces the scalar loop of NB03 (oracle, L1 full, first 1.5 s of the 1.6 s cycle)
def fk(q): return fk_b(q[None])[0]
def _scalar_nb03(R, alpha, n):
    q = R["q"][0].copy(); qd = R["qd"][0].copy(); I = np.zeros(3); mem = MemL1(alpha); mem.reset(n//STEP_C + 2); Hp = MemL1(alpha, dt); Hp.reset(n); mm = np.zeros(3); P = []
    for i in range(n):
        p = fk(q); P.append(p); Dq = Hp.push(q)
        if i % STEP_C == 0: mm = B_FRAC*mem.push(q)
        e = R["q"][i] - q; ed = R["qd"][i] - qd; I += e*dt; v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*I
        tau = np.clip(F_oracle_b(q[None], qd[None], v[None], PAYLOAD, p[None])[0] + mm, -TAU_MAX, TAU_MAX)
        qd = qd + plant_qdd_b(q[None], qd[None], tau[None], p[None], PAYLOAD, Dq[None])[0]*dt; q = q + qd*dt
    return 1000*np.sqrt(np.mean(np.linalg.norm(R["P"][:n] - np.array(P), axis=1)**2))
_R = reference(1.6); _n = 3000; _Rs = {k: (v[:_n] if isinstance(v, np.ndarray) else v) for k, v in _R.items()}; _Rs["N"] = _n
_b = simulate_batch(F_oracle_b, 0.5, np.full(3, B_FRAC), [("L1 full", "l1", {}), ("none", "none", {})], _Rs, 0.5)
print("batch vs scalar loop (oracle, L1 full, 1.5 s): RMSE %.6f vs %.6f mm" % (_b[0]["Cartesian RMSE (mm)"], _scalar_nb03(_R, 0.5, _n)))

## E2: closed loop with the SOE realization (NB06 models, scenarios and code; resumable)

`L1 full` is the reference and `Oustaloup 7 [1e-2, 1e3]` the NB06 recommendation; the SOE rows are new. The PINN (no memory) is rerun for the gain-kept metric.

In [ ]:
key = lambda d: (float(d["alpha true"]), d["Model"], int(d["Seed"]), d["Scenario"])
rows = pd.read_csv(OUT_CL).to_dict("records") if os.path.exists(OUT_CL) else []
done = {key(r) for r in rows}
jobs = [(Mo, sc) for Mo in MODELS for sc in REFS if key({"alpha true": Mo[0], "Model": Mo[1], "Seed": Mo[2], "Scenario": sc}) not in done]
steps = sum(REFS[sc]["N"]*(0.3 + 0.7*len(Mo[6])/len(REALIZ)) for Mo, sc in jobs)
print(f"{len(jobs)} batches to run ({len(done)} already in {OUT_CL})")
t_all = time.time(); s_done = 0
for k, ((alpha, label, seed, F, a_hat, B_hat, realiz), sc) in enumerate(jobs, 1):
    t0 = time.time(); res = simulate_batch(F, a_hat, B_hat, realiz, REFS[sc], alpha)
    for (rl, _, _), r in zip(realiz, res):
        rows.append({"alpha true": alpha, "Model": label, "Seed": seed, "Scenario": sc, "Realization": rl, "alpha hat": a_hat,
                     "B hat mean": float(np.mean(B_hat)), "Batch time (s)": time.time() - t0, "Unstable": r is None, **(r or {})})
    pd.DataFrame(rows).to_csv(OUT_CL, index=False)
    s_done += REFS[sc]["N"]*(0.3 + 0.7*len(realiz)/len(REALIZ)); el = time.time() - t_all
    l1 = next((r for (rl, _, _), r in zip(realiz, res) if rl in ("L1 full", "none")), None)
    print(f"[{k}/{len(jobs)}] α {alpha} | {label:16s} s{seed} | {sc:20s} | {len(realiz):2d} realiz. | "
          + (f"RMSE ({'L1' if len(realiz) > 1 else 'no memory'}) {1e3*l1['Cartesian RMSE (mm)']:.1f} µm" if l1 else "UNSTABLE")
          + f" | {time.time() - t0:.0f} s | elapsed {el/60:.1f} min, left ≈ {el/s_done*(steps - s_done)/60:.0f} min", flush=True)
cl = pd.DataFrame(rows)
print(f"closed loop: {(time.time() - t_all)/60:.1f} min | unstable: {int(cl['Unstable'].astype(bool).sum())} of {len(cl)}")

# consistency with NB06 (same plant, controller, weights and code): L1 full, Oustaloup 7 and PINN rows must coincide
if os.path.exists(NB06_CL_CSV) and len(cl):
    o = pd.read_csv(NB06_CL_CSV); k6 = ["alpha true", "Model", "Seed", "Scenario", "Realization"]
    m = cl.merge(o[k6 + ["Cartesian RMSE (mm)"]], on=k6, suffixes=("", " NB06"))
    if len(m):
        rd = ((m["Cartesian RMSE (mm)"] - m["Cartesian RMSE (mm) NB06"]).abs()/m["Cartesian RMSE (mm) NB06"]); same = ~m.Realization.str.endswith("+ tail")
        if same.any(): print(f"NB07 vs NB06, unchanged code ({int(same.sum())} runs: {sorted(m[same].Realization.unique())}): max relative difference {rd[same].max():.1e}  (should be < 1e-9)")
        if (~same).any():
            m["Δ RMSE corrected − NB06 (µm)"] = 1e3*(m["Cartesian RMSE (mm)"] - m["Cartesian RMSE (mm) NB06"])
            print("window + tail, corrected code (NB07) − NB06 code, mean over seeds (µm):")
            print(m[~same].groupby(["alpha true", "Model", "Scenario", "Realization"])["Δ RMSE corrected − NB06 (µm)"].mean().round(3).to_string())

## Summary: tracking loss of each realization against the L1 sum, and share of the fPINN gain over the PINN that it keeps

For each α and scenario: mean Cartesian RMSE over seeds, paired difference with `L1 full` (µm and %, paired t-test over seeds), and **gain kept** = (RMSE_PINN − RMSE_realization) / (RMSE_PINN − RMSE_L1) over the same seeds (100 % = as good as full memory, 0 % = no better than the integer PINN).

In [ ]:
ok = cl[~cl["Unstable"].astype(bool)].copy(); summ = []
for (alpha, sc), g in ok.groupby(["alpha true", "Scenario"]):
    pinn = g[(g.Model == "PINN")].set_index("Seed")["Cartesian RMSE (mm)"]
    for model in [m for m in g.Model.unique() if m != "PINN"]:
        gm = g[g.Model == model]; l1 = gm[gm.Realization == "L1 full"].set_index("Seed")["Cartesian RMSE (mm)"]
        for rl in [r[0] for r in REALIZ]:
            x = gm[gm.Realization == rl].set_index("Seed")["Cartesian RMSE (mm)"]
            if not len(x): continue
            s = x.index.intersection(l1.index); d = 1e3*(x[s] - l1[s])
            row = {"alpha true": alpha, "Scenario": sc, "Model": model, "Realization": rl, "n seeds": len(x),
                   "RMSE mean (µm)": 1e3*x.mean(), "RMSE std (µm)": 1e3*x.std(), "Δ vs L1 (µm)": d.mean(), "Δ vs L1 (%)": 100*(d/(1e3*l1[s])).mean(),
                   "p (paired t, vs L1)": stats.ttest_rel(x[s], l1[s]).pvalue if len(s) >= 3 and np.any(d != 0) else np.nan,
                   "RMSE last cycle (µm)": 1e3*gm[gm.Realization == rl]["RMSE last cycle (mm)"].mean(),
                   "Memory torque diff vs L1 (mN m)": gm[gm.Realization == rl]["Memory torque diff vs L1 RMS (mN m)"].mean()}
            sp = s.intersection(pinn.index)
            if len(sp) and model != "oracle (frac)":
                row["PINN RMSE (µm)"] = 1e3*pinn[sp].mean()
                row["gain kept (%)"] = 100*((pinn[sp] - x[sp]).sum()/(pinn[sp] - l1[sp]).sum()) if (pinn[sp] - l1[sp]).sum() > 0 else np.nan
            summ.append(row)
summ = pd.DataFrame(summ); summ.to_csv(f"NB07_fPINN_Summary{SUFFIX}.csv", index=False)
cols = ["Realization", "RMSE mean (µm)", "Δ vs L1 (µm)", "Δ vs L1 (%)", "p (paired t, vs L1)", "gain kept (%)", "RMSE last cycle (µm)", "Memory torque diff vs L1 (mN m)"]
for (alpha, sc, model), g in summ.groupby(["alpha true", "Scenario", "Model"]):
    print(f"\nα = {alpha} | {sc} | {model}" + (f" | PINN {g['PINN RMSE (µm)'].iloc[0]:.1f} µm" if "PINN RMSE (µm)" in g and g["PINN RMSE (µm)"].notna().any() else ""))
    print(g[[c for c in cols if c in g]].round(3).to_string(index=False))

## E3a: cost per update in Python against the elapsed operation time, including the exact fast L1

The naive L1 is timed with a history of `t/T_C` samples already stored (as NB06). The block-FFT L1 is timed step by step over two full blocks after the same history: the **mean** is its amortized cost and the **max** is the step that recomputes the FFT of the whole history, which a real-time loop must also fit in the period (or spread over a background task). Fixed-cost realizations do not depend on t. Element-wise NumPy and `einsum` only (no BLAS calls), FFT by `scipy.fft` (pocketfft, no MKL).

In [ ]:
def time_push(mem, n_hist, reps):
    mem.reset(max(n_hist, 1) + reps + 2); rng = np.random.default_rng(0)
    if isinstance(mem, MemL1): mem.q_last = np.zeros(3); mem.n = n_hist; mem.dq[1:n_hist + 1] = 1e-4*rng.standard_normal((n_hist, 3))
    else:
        for _ in range(min(n_hist, 2000)): mem.push(1e-2*rng.standard_normal(3))
    qs = 1e-2*rng.standard_normal((reps, 3)); t0 = time.perf_counter()
    for v in qs: mem.push(v)
    return 1e6*(time.perf_counter() - t0)/reps
def time_block(L, n_hist, alpha=0.5):
    """per-step times [µs] of MemL1Block over 2 blocks after n_hist (multiple of L) stored increments."""
    n_hist = (n_hist//L)*L; mem = MemL1Block(alpha, L); mem.reset(n_hist + 2*L + 2); rng = np.random.default_rng(0)
    mem.q_last = np.zeros(3); mem.n = n_hist; mem.dq[1:n_hist + 1] = 1e-4*rng.standard_normal((n_hist, 3))
    qs = 1e-2*rng.standard_normal((2*L, 3)); ts = np.empty(2*L)
    for i, v in enumerate(qs): t0 = time.perf_counter(); mem.push(v); ts[i] = time.perf_counter() - t0
    return 1e6*ts, n_hist
TIM_FIXED = [("window 0.5 s + tail", "win", dict(Tw=0.5, tail="mean")), ("window 2 s + tail", "win", dict(Tw=2.0, tail="mean")),
             ("Oustaloup 7 [1e-2, 1e3]", "oust", dict(n=7, wb=1e-2, wh=1e3)), ("Oustaloup 9 [1e-4, 1e3]", "oust", dict(n=9, wb=1e-4, wh=1e3))]
TIM_FIXED += [(f"SOE {fmt_eps(e)} [{fmt_hor(SOE_HOR)}]", "soe", dict(eps=e, T=SOE_HOR)) for e in SOE_EPS]
_lim, _mkl_prev = None, None                      # one BLAS thread (see NB06): threadpoolctl, else mkl-service, else a warning
try:
    from threadpoolctl import threadpool_limits; _lim = threadpool_limits(1); print("BLAS limited to 1 thread (threadpoolctl)")
except Exception as _e:
    try:
        import mkl; _mkl_prev = mkl.get_max_threads(); mkl.set_num_threads(1); print("BLAS limited to 1 thread with mkl-service")
    except Exception: print(f"could not limit BLAS threads ({type(_e).__name__}); the kernels here make no BLAS calls, so the times are not affected")
tim = []; t_tim = time.time()
for T in TIMING_T:
    n = int(T/T_C)
    us = min(time_push(MemL1(0.5), n, 200 if n < 300000 else 60) for _ in range(3)); fl, by = cost("l1", {}, n=n)
    tim.append({"Elapsed time (s)": T, "Realization": "L1 full (naive)", "µs per update, mean": us, "µs per update, max": np.nan, "flops per update": fl, "RAM (kB)": by/1024})
    for L in BLOCK_L:
        ts, nh = time_block(L, n); _, by = cost("l1block", dict(L=L), n=n)
        tim.append({"Elapsed time (s)": T, "Realization": f"L1 full (block FFT, L = {L})", "µs per update, mean": ts.mean(), "µs per update, max": ts.max(),
                    "µs per update, median": np.median(ts), "flops per update": np.nan, "RAM (kB)": by/1024})
    for label, kind, kw in TIM_FIXED:
        if T == TIMING_T[0]: us = min(time_push(make_mem(kind, 0.5, **kw), n, 2000) for _ in range(3))
        else: us = next(r["µs per update, mean"] for r in tim if r["Realization"] == label)
        fl, by = cost(kind, kw, 0.5)
        tim.append({"Elapsed time (s)": T, "Realization": label, "µs per update, mean": us, "µs per update, max": np.nan, "flops per update": fl, "RAM (kB)": by/1024})
    print(f"t = {T:5g} s done ({time.time() - t_tim:.0f} s)", flush=True)
if _lim is not None: _lim.restore_original_limits()
if _mkl_prev is not None: mkl.set_num_threads(_mkl_prev)
tim = pd.DataFrame(tim); tim.to_csv(f"NB07_fPINN_Timing{SUFFIX}.csv", index=False)
print("mean µs per update (Python):"); print(tim.pivot(index="Realization", columns="Elapsed time (s)", values="µs per update, mean").round(1).to_string())
print("\nworst step µs (block FFT):"); print(tim.dropna(subset=["µs per update, max"]).pivot(index="Realization", columns="Elapsed time (s)", values="µs per update, max").round(0).to_string())
print("\nRAM (kB):"); print(tim.pivot(index="Realization", columns="Elapsed time (s)", values="RAM (kB)").round(1).to_string())

## E3b: the kernels in plain C99 (naive L1, window + tail, Oustaloup, SOE)

Writes `NB07_fPINN_memkernels.c` (the NB06 kernels plus the SOE update; the SOE nodes for α = 0.5 are written into the file) with a benchmark `main` that prints a CSV of ns per update. It looks for a C compiler (`gcc`, `clang`, `cc`); if one is found the file is compiled with `-O2`, checked against the Python realizations on the same input and run; **if none is found the step is skipped** and the file is left for any machine with a compiler (`gcc -O2 -std=gnu99 NB07_fPINN_memkernels.c -o memk -lm && ./memk > NB07_fPINN_TimingC.csv`). The block-FFT L1 is not written in C (it needs an FFT library); its Python times are in E3a.

In [ ]:
def c_array(v): return "{" + ", ".join(f"{x:.17e}" for x in v) + "}"
_soe_c = []
for e in SOE_EPS:
    s = MemSOE(0.5, e, SOE_HOR); _soe_c.append((fmt_eps(e), s.nm, s.E[:, 0], s.G[:, 0], s.w))
SOE_DEFS = "\n".join(f"static const double SOE{i}_E[] = {c_array(E)};\nstatic const double SOE{i}_G[] = {c_array(G)};\nstatic const double SOE{i}_W[] = {c_array(W)};"
                     for i, (_, nm, E, G, W) in enumerate(_soe_c))
SOE_TABLE = "static const SoeDef SOES[] = {" + ", ".join(f'{{"{lab}", {nm}, SOE{i}_E, SOE{i}_G, SOE{i}_W}}' for i, (lab, nm, _, _, _) in enumerate(_soe_c)) + "};\n#define N_SOE " + str(len(_soe_c))
C_SRC = r'''
/* NB07_fPINN_memkernels.c - memory term D^alpha q of the fPINN (3 joints, double): full L1 (naive), window + tail,
   Oustaloup (parallel Tustin modes) and sum of exponentials (SOE). Update functions use no dynamic memory.
   Build: gcc -O2 -std=gnu99 NB07_fPINN_memkernels.c -o memk -lm */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#define NJ 3
static double now_s(void){ struct timespec t; clock_gettime(CLOCK_MONOTONIC, &t); return t.tv_sec + 1e-9*t.tv_nsec; }
static double l1w(double a, long k){ return pow(k + 1.0, 1 - a) - pow((double)k, 1 - a); }

/* ---- full-memory L1 (naive): D = c * sum_{k=0}^{n-1} b_k dq_{n-k} ---- */
typedef struct { double c, *b, *dq, ql[NJ]; long n; int started; } L1full;
void l1_update(L1full *s, const double q[NJ], double D[NJ]) {
    int j; long k;
    if (!s->started) { for (j = 0; j < NJ; j++) { s->ql[j] = q[j]; D[j] = 0; } s->started = 1; return; }
    s->n++; for (j = 0; j < NJ; j++) { s->dq[NJ*s->n + j] = q[j] - s->ql[j]; s->ql[j] = q[j]; }
    for (j = 0; j < NJ; j++) { double acc = 0; for (k = 0; k < s->n; k++) acc += s->b[k]*s->dq[NJ*(s->n - k) + j]; D[j] = s->c*acc; }
}
/* ---- window of Nw increments + O(1) tail correction (mean-value form) ---- */
#define NW_MAX 5000
typedef struct { double c, a, bw[NW_MAX + 1], buf[NJ][NW_MAX], qb[NJ][NW_MAX + 1], q0[NJ], ql[NJ], ssum[NJ]; long n; int Nw, started, tail; } Win;
void win_update(Win *s, const double q[NJ], double D[NJ]) {
    int j, k, Nw = s->Nw;
    if (!s->started) { for (j = 0; j < NJ; j++) { s->q0[j] = s->ql[j] = q[j]; s->qb[j][0] = 0; D[j] = 0; s->ssum[j] = 0; } s->started = 1; return; }
    s->n++; long n = s->n; int i = (int)((n - 1) % Nw); int m = n < Nw ? (int)n : Nw;
    for (j = 0; j < NJ; j++) {
        s->buf[j][i] = q[j] - s->ql[j]; s->ql[j] = q[j];
        double acc = 0; for (k = 0; k < m; k++) { int idx = i - k; if (idx < 0) idx += Nw; acc += s->bw[k]*s->buf[j][idx]; }
        if (s->tail && n > Nw) {
            if (n - Nw - 1 >= 1) s->ssum[j] += s->qb[j][(n - Nw - 1) % (Nw + 1)];   /* read q~_{n-Nw-1} before its slot is overwritten */
            acc += s->bw[Nw]*s->qb[j][(n - Nw) % (Nw + 1)];
            if (n - Nw - 1 >= 1) acc -= (s->bw[Nw] - l1w(s->a, n - 1))*s->ssum[j]/(double)(n - Nw - 1);
        }
        s->qb[j][n % (Nw + 1)] = q[j] - s->q0[j];
        D[j] = s->c*acc;
    }
}
/* ---- Oustaloup, parallel first-order modes (Tustin) ---- */
#define NM_MAX 15
typedef struct { int nm; double K, r[NM_MAX], A[NM_MAX], Bd[NM_MAX], X[NM_MAX][NJ], up[NJ], q0[NJ]; int started; } Oust;
void oust_update(Oust *s, const double q[NJ], double D[NJ]) {
    int j, k; if (!s->started) { for (j = 0; j < NJ; j++) { s->q0[j] = q[j]; s->up[j] = 0; } s->started = 1; }
    for (j = 0; j < NJ; j++) {
        double u = q[j] - s->q0[j], acc = u;
        for (k = 0; k < s->nm; k++) { s->X[k][j] = s->A[k]*s->X[k][j] + s->Bd[k]*(u + s->up[j]); acc += s->r[k]*s->X[k][j]; }
        s->up[j] = u; D[j] = s->K*acc;
    }
}
void oust_init(Oust *s, double a, int nm, double wb, double wh, double h) {
    int N = (nm - 1)/2, i, k; double p[NM_MAX], z[NM_MAX];
    s->nm = nm; s->K = pow(wh, a); s->started = 0;
    for (i = 0; i < nm; i++) { int kk = i - N; p[i] = wb*pow(wh/wb, (kk + N + (1 + a)/2)/(2.0*N + 1)); z[i] = wb*pow(wh/wb, (kk + N + (1 - a)/2)/(2.0*N + 1)); }
    for (i = 0; i < nm; i++) { double num = 1, den = 1; for (k = 0; k < nm; k++) { num *= z[k] - p[i]; if (k != i) den *= p[k] - p[i]; }
        s->r[i] = num/den; s->A[i] = (2 - p[i]*h)/(2 + p[i]*h); s->Bd[i] = h/(2 + p[i]*h); for (k = 0; k < NJ; k++) s->X[i][k] = 0; }
}
/* ---- sum of exponentials: S_i <- E_i S_i + G_i dq_{n-1};  D = c dq_n + sum_i w_i S_i ---- */
#define NS_MAX 80
typedef struct { const char *label; int nm; const double *E, *G, *W; } SoeDef;
typedef struct { int nm; double c; const double *E, *G, *W; double S[NS_MAX][NJ], dp[NJ], ql[NJ]; int started; } Soe;
void soe_init(Soe *s, const SoeDef *d, double c) { int i, j; s->nm = d->nm; s->c = c; s->E = d->E; s->G = d->G; s->W = d->W; s->started = 0;
    for (i = 0; i < NS_MAX; i++) for (j = 0; j < NJ; j++) s->S[i][j] = 0; for (j = 0; j < NJ; j++) s->dp[j] = 0; }
void soe_update(Soe *s, const double q[NJ], double D[NJ]) {
    int j, i; if (!s->started) { for (j = 0; j < NJ; j++) { s->ql[j] = q[j]; D[j] = 0; } s->started = 1; return; }
    for (j = 0; j < NJ; j++) {
        double d = q[j] - s->ql[j], acc = s->c*d; s->ql[j] = q[j];
        for (i = 0; i < s->nm; i++) { s->S[i][j] = s->E[i]*s->S[i][j] + s->G[i]*s->dp[j]; acc += s->W[i]*s->S[i][j]; }
        s->dp[j] = d; D[j] = acc;
    }
}
@SOE_DEFS@
@SOE_TABLE@
int main(void) {
    double a = 0.5, h = 0.002, D[NJ], q[NJ], sink = 0; long hist[] = {500, 5000, 30000, 300000, 600000, 1800000}; int nh = 6, t, j, k;
    double c = pow(h, -a)/tgamma(2 - a);
    /* self-check input: q_j(t) = sin(t), 0.3 cos(2t), t/6 on 0..6 s; prints D at t = 6 s for the Python check */
    { long n = 3000, i; L1full L; L.c = c; L.n = 0; L.started = 0; L.b = malloc(sizeof(double)*(n + 2)); L.dq = calloc(NJ*(n + 2), sizeof(double));
      static Win W; W.c = c; W.a = a; W.Nw = 250; W.tail = 1; W.n = 0; W.started = 0; for (i = 0; i <= W.Nw; i++) W.bw[i] = l1w(a, i);
      Oust O; oust_init(&O, a, 7, 1e-2, 1e3, h); static Soe S[N_SOE]; double D1[NJ], D2[NJ], D3[NJ], D4[N_SOE][NJ];
      for (k = 0; k < N_SOE; k++) soe_init(&S[k], &SOES[k], c);
      for (i = 0; i <= n; i++) L.b[i] = l1w(a, i);
      for (i = 0; i <= n; i++) { double tt = 6.0*i/n; q[0] = sin(tt); q[1] = 0.3*cos(2*tt); q[2] = tt/6;
          l1_update(&L, q, D1); win_update(&W, q, D2); oust_update(&O, q, D3); for (k = 0; k < N_SOE; k++) soe_update(&S[k], q, D4[k]); }
      printf("check,L1,%.12e,%.12e,%.12e\n", D1[0], D1[1], D1[2]); printf("check,win0.5+tail,%.12e,%.12e,%.12e\n", D2[0], D2[1], D2[2]);
      printf("check,oust7,%.12e,%.12e,%.12e\n", D3[0], D3[1], D3[2]);
      for (k = 0; k < N_SOE; k++) printf("check,soe%s,%.12e,%.12e,%.12e\n", SOES[k].label, D4[k][0], D4[k][1], D4[k][2]);
      free(L.b); free(L.dq); }
    printf("method,size,elapsed_s,ns_per_update\n");
    for (t = 0; t < nh; t++) {
        long n = hist[t], i, reps = n > 100000 ? 50 : 500; L1full L; L.c = c; L.started = 1; L.n = n;
        L.b = malloc(sizeof(double)*(n + reps + 2)); L.dq = calloc(NJ*(n + reps + 2), sizeof(double));
        for (i = 0; i < n + reps + 2; i++) L.b[i] = l1w(a, i);
        for (i = 0; i < NJ*(n + 1); i++) L.dq[i] = 1e-4*((i*7919) % 1000 - 500)/500.0;
        for (j = 0; j < NJ; j++) L.ql[j] = 0;
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*i; q[1] = -q[0]; q[2] = 0.5*q[0]; l1_update(&L, q, D); sink += D[0]; }
        printf("L1 full (naive),%ld,%.1f,%.1f\n", n, n*h, 1e9*(now_s() - t0)/reps); free(L.b); free(L.dq);
    }
    { int ws[] = {250, 1000}, w; for (w = 0; w < 2; w++) { static Win W; long i, reps = 200000; W.c = c; W.a = a; W.Nw = ws[w]; W.tail = 1; W.n = 0; W.started = 0;
        for (i = 0; i <= W.Nw; i++) W.bw[i] = l1w(a, i);
        for (i = 0; i < 5000; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; win_update(&W, q, D); }
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; win_update(&W, q, D); sink += D[0]; }
        printf("window %g s + tail,%d,-,%.1f\n", W.Nw*h, W.Nw, 1e9*(now_s() - t0)/reps); } }
    { int ms[] = {5, 7, 9, 11}; double wbs[] = {1e-2, 1e-2, 1e-4, 1e-4}; for (k = 0; k < 4; k++) { Oust O; long i, reps = 1000000; oust_init(&O, a, ms[k], wbs[k], 1e3, h);
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; oust_update(&O, q, D); sink += D[0]; }
        printf("Oustaloup %d [%g; 1e3],%d,-,%.1f\n", ms[k], wbs[k], ms[k], 1e9*(now_s() - t0)/reps); } }
    for (k = 0; k < N_SOE; k++) { static Soe S; long i, reps = 1000000; soe_init(&S, &SOES[k], c);
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; soe_update(&S, q, D); sink += D[0]; }
        printf("SOE %s [1 h],%d,-,%.1f\n", SOES[k].label, SOES[k].nm, 1e9*(now_s() - t0)/reps); }
    fprintf(stderr, "sink %g\n", sink); return 0;
}
'''.replace("@SOE_DEFS@", SOE_DEFS).replace("@SOE_TABLE@", SOE_TABLE)
with open("NB07_fPINN_memkernels.c", "w") as f: f.write(C_SRC)
cc = next((shutil.which(x) for x in ("gcc", "clang", "cc") if shutil.which(x)), None)
timc = None
if cc is None: print("no C compiler found (gcc, clang, cc): E3b skipped; NB07_fPINN_memkernels.c written for another machine.")
else:
    exe = os.path.abspath("NB07_fPINN_memk" + (".exe" if os.name == "nt" else ""))
    r = subprocess.run([cc, "-O2", "-std=gnu99", "NB07_fPINN_memkernels.c", "-o", exe, "-lm"], capture_output=True, text=True)
    if r.returncode: print(f"compilation with {cc} failed, E3b skipped:", r.stderr[:1500])
    else:
        out = subprocess.run([exe], capture_output=True, text=True).stdout.strip().splitlines()
        _x = np.column_stack([np.sin(np.linspace(0, 6, 3001)), 0.3*np.cos(2*np.linspace(0, 6, 3001)), np.linspace(0, 6, 3001)/6])
        py = {"L1": online(MemL1(0.5), _x)[-1], "win0.5+tail": online(MemWindow(0.5, 0.5, "mean"), _x)[-1], "oust7": online(MemOustaloup(0.5, 7, 1e-2, 1e3), _x)[-1]}
        py.update({f"soe{fmt_eps(e)}": online(MemSOE(0.5, e, SOE_HOR), _x)[-1] for e in SOE_EPS})
        for c_ in [l.split(",") for l in out if l.startswith("check")]:
            print(f"C vs Python, {c_[1]:12s}: max rel. diff {np.abs(np.array(c_[2:], float) - py[c_[1]]).max()/np.abs(py[c_[1]]).max():.1e}")
        from io import StringIO
        timc = pd.read_csv(StringIO("\n".join(l for l in out if not l.startswith("check")))); timc["µs per update"] = timc["ns_per_update"]/1e3
        import platform; timc["CPU"] = platform.processor() or platform.machine(); timc["compiler"] = os.path.basename(cc)
        timc.to_csv(f"NB07_fPINN_TimingC{SUFFIX}.csv", index=False); print(timc.to_string(index=False))

## Figure (quick look; the article figures are redrawn from the CSVs)

In [ ]:
FIG_R = ["window 2 s + tail", "window 0.5 s + tail", "Oustaloup 7 [1e-2, 1e3]", "Oustaloup 9 [1e-4, 1e3]", "SOE 1e-3 [1 h]", "SOE 1e-3 [60 s]"]
fig, ax = plt.subplots(2, 2, figsize=(11, 7.5)); ax = ax.ravel()
for i, nm in enumerate(("Pauses at pick pose", "Away from ready")):
    g = lmin[(lmin.alpha == ALPHA_MAIN) & (lmin.Reference == nm)]
    for rl in FIG_R:
        s = g[g.Realization == rl]
        if len(s): ax[i].plot(s["Bin start (min)"] + BIN_S/120, s["Error max (mN m)"], marker=".", label=rl)
    s = g[g.Realization == "none"]
    if len(s): ax[i].plot(s["Bin start (min)"] + BIN_S/120, s["Memory torque RMS (mN m)"], "k:", label="L1 memory torque (RMS)")
    ax[i].set_yscale("log"); ax[i].set_xlabel("operation time (min)"); ax[i].set_ylabel("max memory torque error per bin (mN m)")
    ax[i].set_title(f"({'ab'[i]}) {nm}, α = {ALPHA_MAIN}", loc="left", fontsize=9); ax[i].grid(True, which="both", alpha=0.3)
ax[0].legend(fontsize=7)
# (c) step error of Oustaloup per unit displacement
tau = np.logspace(-2, 4.5, 400)
for n, wb in ((7, 1e-2), (9, 1e-4)):
    ax[2].plot(tau, 1e3*B_FRAC*np.abs(QT_PICK[j3])*np.abs(oust_step_err(ALPHA_MAIN, n, wb, 1e3, tau)), label=f"Oustaloup {n} [{fmt_eps(wb)}, 1e3]")
    ax[2].axhline(1e3*B_FRAC*wb**ALPHA_MAIN*abs(QT_PICK[j3]), ls=":", lw=1, color=ax[2].lines[-1].get_color())
ax[2].plot(tau, 1e3*B_FRAC*np.abs(QT_PICK[j3])*tau**(-ALPHA_MAIN)/Gamma(1 - ALPHA_MAIN), "k--", lw=1, label="exact response (L1 limit)")
ax[2].set_xscale("log"); ax[2].set_yscale("log"); ax[2].set_xlabel("hold time τ (s)"); ax[2].set_ylabel("|error| of a held step (mN m)")
ax[2].set_title(f"(c) held displacement q̃ = {abs(QT_PICK[j3]):.2f} rad; dotted: limit B·ω_b^α·q̃", loc="left", fontsize=9); ax[2].legend(fontsize=7); ax[2].grid(True, which="both", alpha=0.3)
# (d) cost per update
for lab in ["L1 full (naive)", "L1 full (block FFT, L = 1024)", "window 0.5 s + tail", "Oustaloup 7 [1e-2, 1e3]", "SOE 1e-3 [1 h]"]:
    g = tim[tim.Realization == lab]
    if len(g): l, = ax[3].plot(g["Elapsed time (s)"], g["µs per update, mean"], marker="o", label=lab + " (Python, mean)")
    if lab.startswith("L1 full (block") and len(g): ax[3].plot(g["Elapsed time (s)"], g["µs per update, max"], marker="v", ls="--", color=l.get_color(), label=lab + " (Python, worst step)")
if timc is not None:
    gc_ = timc[timc.method == "L1 full (naive)"]; ax[3].plot(gc_["size"]*T_C, gc_["µs per update"], marker="s", ls="-.", color="k", label="L1 full (naive, C)")
    for m in ("Oustaloup 7 [0.01; 1e3]", "SOE 1e-3 [1 h]"):
        v = timc[timc.method == m]["µs per update"]
        if len(v): ax[3].axhline(v.iloc[0], ls="-.", lw=0.8, color="0.4"); ax[3].text(TIMING_T[0], v.iloc[0]*1.1, m + " (C)", fontsize=6, color="0.4")
ax[3].axhline(1e6*T_C, color="r", ls=":", lw=1); ax[3].text(TIMING_T[-1], 1.1e6*T_C, "control period 2 ms", ha="right", fontsize=7, color="r")
ax[3].set_xscale("log"); ax[3].set_yscale("log"); ax[3].set_xlabel("elapsed operation time (s)"); ax[3].set_ylabel("µs per update")
ax[3].set_title("(d) cost per update on this PC", loc="left", fontsize=9); ax[3].legend(fontsize=6, loc="center right"); ax[3].grid(True, which="both", alpha=0.3)
fig.tight_layout(); fig.savefig(f"fig_nb07_horizonte_largo{SUFFIX}.png", dpi=200); fig.savefig(f"fig_nb07_horizonte_largo{SUFFIX}.pdf"); plt.show()